In [ ]:
QUERY = """
(
all:chatbot OR
all:"artificial intelligence" OR
all:"generative AI" OR
all:GenAI OR
all:"large language model" OR
all:LLM OR
all:LLMs
)
AND
(
all:loneliness OR
all:"social isolation" OR
all:"social alienation" OR
all:"social exclusion" OR
all:"social withdrawal" OR
all:"social connection" OR
all:"social inclusion" OR
all:belonging OR
all:"social satisfaction" OR
all:"perceived cohesion" OR
all:"social well-being" OR
all:"social wellbeing" OR
all:"interpersonal well-being" OR
all:"interpersonal wellbeing"
)
"""


In [ ]:
import arxiv
import pandas as pd

client = arxiv.Client(
    page_size=100,
    delay_seconds=5,
    num_retries=10
)

search = arxiv.Search(
    query=QUERY,
    max_results=10000,
    sort_by=arxiv.SortCriterion.Relevance
)

papers = []

for paper in client.results(search):

    papers.append({
        "title": paper.title,
        "authors": "; ".join(a.name for a in paper.authors),
        "published": paper.published,
        "updated": paper.updated,
        "summary": paper.summary.replace("\n", " "),
        "categories": "; ".join(paper.categories),
        "pdf_url": paper.pdf_url,
        "entry_id": paper.entry_id,
    })

df = pd.DataFrame(papers)

print(f"Found {len(df)} papers")

df.to_csv(
    "ai_social_connection_arxiv_10000.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved to arxiv.csv")


In [ ]:
import requests
import re


LLM_CHOICES = [
    "chatbot",
    "chatbots",
    "artificial intelligence",
    "generative AI",
    "GenAI",
    "large language model",
    "large language models",
    "LLM",
    "LLMs",
]

HEALTH_OUTCOMES = [
    "loneliness",
    "social isolation",
    "social alienation",
    "social exclusion",
    "social withdrawal",
    "social connection",
    "social inclusion",
    "belonging",
    "social satisfaction",
    "perceived cohesion",
    "social well-being",
    "social wellbeing",
    "interpersonal well-being",
    "interpersonal wellbeing",
]


def is_multi_word(term):
    return len(term.strip().split()) > 1

def get_anchor(term):
    if is_multi_word(term):
        return term.strip().split()[-1].lower()
    return term.strip().lower()

def exact_match(text, term):
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return bool(re.search(pattern, text.lower()))

def get_searchable_text(paper):
    attrs = paper.get("attributes", {})
    title = attrs.get("title", "") or ""
    description = attrs.get("description", "") or ""
    tags = " ".join(attrs.get("tags", []))
    return f"{title} {description} {tags}"

def get_contributors(paper_id):
    print(f"    [contributors] Fetching for {paper_id}...")
    url = f"https://api.osf.io/v2/preprints/{paper_id}/contributors/"
    response = requests.get(url)
    if response.status_code != 200:
        return []
    contributors = []
    for c in response.json().get("data", []):
        name = (c.get("embeds", {})
                 .get("users", {})
                 .get("data", {})
                 .get("attributes", {})
                 .get("full_name"))
        if not name:
            name = c.get("attributes", {}).get("full_name", "Unknown")
        contributors.append(name)
    return contributors

def extract_info(paper):
    attrs = paper.get("attributes", {})
    links = paper.get("links", {})
    rels = paper.get("relationships", {})
    subjects = []
    for path in attrs.get("subjects", []):
        if path:
            subjects.append(" > ".join(s["text"] for s in path))
    return {
        "id": paper.get("id"),
        "title": attrs.get("title"),
        "description": attrs.get("description"),
        "date_published": attrs.get("date_published", "")[:10],
        "doi": links.get("preprint_doi") or attrs.get("doi"),
        "url": links.get("html"),
        "tags": attrs.get("tags", []),
        "subjects": subjects,
        "reviews_state": attrs.get("reviews_state"),
        "provider": rels.get("provider", {}).get("data", {}).get("id"),
    }


def fetch_by_anchor(anchor):
    print(f"    [api] Fetching anchor '{anchor}'...")
    url = "https://api.osf.io/v2/preprints/"
    params = {
        "provider": "socarxiv",
        "filter[description]": anchor,
        "page[size]": 100
    }
    results = []
    page = 1
    while url:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"    [error] Status {response.status_code} for '{anchor}' page {page}")
            break
        data = response.json()
        batch = data.get("data", [])
        results.extend(batch)
        print(f"    [api]   Page {page}: {len(batch)} papers (total: {len(results)})")
        url = data.get("links", {}).get("next")
        params = {}
        page += 1
    return results


def search_and_verify(term_list, label):
    print(f"\n{'='*60}")
    print(f"Searching {label} terms...")
    print(f"{'='*60}")

    all_papers = {}
    term_to_ids = {}


    anchor_to_terms = {}
    for term in term_list:
        anchor = get_anchor(term)
        anchor_to_terms.setdefault(anchor, []).append(term)

    print(f"\nAnchors to fetch: {list(anchor_to_terms.keys())}")

    for anchor, terms in anchor_to_terms.items():
        print(f"\n  Anchor '{anchor}' covers terms: {terms}")


        papers = fetch_by_anchor(anchor)
        print(f"  → {len(papers)} papers retrieved for anchor '{anchor}'")


        for term in terms:
            matched_ids = set()
            for paper in papers:
                pid = paper.get("id")
                text = get_searchable_text(paper)

                if is_multi_word(term):

                    if exact_match(text, term):
                        matched_ids.add(pid)
                        all_papers[pid] = paper

                else:

                    matched_ids.add(pid)
                    all_papers[pid] = paper

            term_to_ids[term] = matched_ids
            print(f"  → Term '{term}': {len(matched_ids)} papers after exact match check")

    return term_to_ids, all_papers


def pairwise_match(llm_term_to_ids, health_term_to_ids, all_papers):
    print(f"\n{'='*60}")
    print(f"Pairwise matching: each LLM term × each HEALTH_OUTCOME term")
    print(f"{'='*60}")


    paper_results = {}

    for llm_term, llm_ids in llm_term_to_ids.items():
        for health_term, health_ids in health_term_to_ids.items():
            overlap = llm_ids & health_ids
            if overlap:
                print(f"  '{llm_term}' × '{health_term}': {len(overlap)} papers")
                for pid in overlap:
                    if pid not in paper_results:
                        paper_results[pid] = {
                            "paper": all_papers[pid],
                            "matched_llm": set(),
                            "matched_health": set(),
                            "pairs": [],
                        }
                    paper_results[pid]["matched_llm"].add(llm_term)
                    paper_results[pid]["matched_health"].add(health_term)
                    paper_results[pid]["pairs"].append((llm_term, health_term))
            else:
                print(f"  '{llm_term}' × '{health_term}': 0 papers")

    return paper_results


print("=" * 60)
print("SocArXiv: LLM × Health Outcome search")
print("=" * 60)


llm_term_to_ids, llm_papers = search_and_verify(LLM_CHOICES, "LLM")


health_term_to_ids, health_papers = search_and_verify(HEALTH_OUTCOMES, "HEALTH_OUTCOME")


all_papers = {**llm_papers, **health_papers}

print(f"\nTotal unique papers found across LLM terms:    {len(llm_papers)}")
print(f"Total unique papers found across health terms:  {len(health_papers)}")


paper_results = pairwise_match(llm_term_to_ids, health_term_to_ids, all_papers)

print(f"\n→ Unique papers matching at least one LLM × health pair: {len(paper_results)}")


In [ ]:
import pandas as pd

rows = []

for pid, result in paper_results.items():
    info = extract_info(result["paper"])
    contribs = get_contributors(pid)

    rows.append({
        "paper_id": pid,
        "title": info["title"],
        "description": info["description"] or "",
        "date_published": info["date_published"],
        "provider": info["provider"],
        "url": info["url"],
        "doi": info["doi"],
        "reviews_state": info["reviews_state"],
        "contributors": "; ".join(contribs) if contribs else "",
        "tags": "; ".join(info["tags"]) if info["tags"] else "",
        "subjects": "; ".join(info["subjects"]) if info["subjects"] else "",
        "matched_llm_terms": "; ".join(sorted(result["matched_llm"])),
        "matched_health_terms": "; ".join(sorted(result["matched_health"])),
        "matched_pairs": "; ".join(
            [f"{llm} × {health}" for llm, health in sorted(set(result["pairs"]))]
        ),
    })

df = pd.DataFrame(rows)

df.to_csv(
    "socarxiv_llm_health_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved {len(df)} papers to socarxiv_llm_health_results.csv")
